In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t001/task.json'
with open(task_file_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from the task.json
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "checkout 错误次数告警" (checkout error count alert). It occurred within the time window from 2026-04-25 13:18:12 to 2026-04-25 13:28:12 (UTC+8). The alert was triggered by the operation `checkout::/oteldemo.CheckoutService/PlaceOrder` of type `apm.operation` in the `apm` domain. The user is requesting help to analyze the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

('checkout 错误次数告警', {'start': '2026-04-25T05:18:12.716735+00:00', 'end': '2026-04-25T05:28:12.716735+00:00'}, {'entity_id': None, 'entity_name': None, 'entity_type': None, 'entity_domain': None}, '<code vibeops_object type="alert_event">checkout 错误次数告警<alert_event event_id="ee53bc56825ddb197fd34ca61136d964" trans_id="tbdh9alum56k56pj9864sdn2i0" rule_id="08493be2-a181-4480-9895-2b2c5607122e" rule_name="checkout 错误次数告警" entity_id="d219413245b68b297976412bbee076cf" entity_name="checkout::/oteldemo.CheckoutService/PlaceOrder" entity_type="apm.operation" entity_domain="apm" alert_time="2026-04-25T13:20:26+08:00" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

# Load the topology.json file to understand the service dependency graph
topology_file_path = 'rca100/cases/t001/topology.json'
with open(topology_file_path, 'r') as file:
    topology_data = json.load(file)

# Extract entities and edges from the topology data
entities = topology_data.get('entities', [])
edges = topology_data.get('edges', [])

# Identify the checkout service and its dependencies
checkout_service_id = None
for entity in entities:
    if entity.get('type') == 'apm.service' and 'checkout' in entity.get('name', '').lower():
        checkout_service_id = entity['id']
        break

# Find upstream and downstream dependencies of the checkout service
upstream_dependencies = []
downstream_dependencies = []
if checkout_service_id:
    for edge in edges:
        if edge['src'] == checkout_service_id and edge['relation'] == 'calls':
            downstream_dependencies.append(edge['dst'])
        elif edge['dst'] == checkout_service_id and edge['relation'] == 'calls':
            upstream_dependencies.append(edge['src'])

# Map dependencies to their names
upstream_names = [entity['name'] for entity in entities if entity['id'] in upstream_dependencies]
downstream_names = [entity['name'] for entity in entities if entity['id'] in downstream_dependencies]

checkout_service_id, upstream_names, downstream_names

```
Out[2]:
```
The `checkout` service (ID: 55e5d48e4d1bdc8b07f350cb89476ad1) has the following dependencies:

- **Upstream dependency**: `frontend` service.
- **Downstream dependencies**: `shipping`, `email`, `cart`, `currency`, `payment`, `product-catalog`, `flagd:8015`, `email:8080`, `orders`, and several specific operations such as `/oteldemo.ShippingService/GetQuote`, `/oteldemo.ShippingService/ShipOrder`, `/oteldemo.PaymentService/Charge`, `/oteldemo.CartService/EmptyCart`, `/oteldemo.CurrencyService/Convert`, `/oteldemo.CartService/GetCart`, and `/oteldemo.ProductCatalogService/GetProduct`.

The original code execution output of IPython Kernel is also provided below for reference:

('55e5d48e4d1bdc8b07f350cb89476ad1', ['frontend'], ['shipping', 'email', 'cart', 'currency', 'payment', 'product-catalog', 'flagd:8015', 'email:8080', 'orders', '/oteldemo.ShippingService/GetQuote', '/oteldemo.ShippingService/ShipOrder', '/oteldemo.PaymentService/Charge', '/oteldemo.CartService/EmptyCart', '/oteldemo.CurrencyService/Convert', '/oteldemo.CartService/GetCart', '/oteldemo.ProductCatalogService/GetProduct'])```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
from pytz import timezone
from datetime import datetime

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t001/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# Define the time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-25T13:18:12.716735+08:00')
alert_end = datetime.fromisoformat('2026-04-25T13:28:12.716735+08:00')

# Convert the time window to Unix microseconds for filtering
alert_start_us = int(alert_start.timestamp() * 1_000_000)
alert_end_us = int(alert_end.timestamp() * 1_000_000)

# Filter metrics for the checkout service and its dependencies
relevant_services = ['checkout', 'frontend', 'shipping', 'email', 'cart', 'currency', 
                     'payment', 'product-catalog', 'flagd:8015', 'email:8080', 'orders']
metrics_filtered = metrics_data[
    metrics_data['entity_name'].str.contains('|'.join(relevant_services), case=False, na=False)
]

# Calculate global thresholds (P95) for all KPIs
global_thresholds = metrics_filtered.groupby(['entity_name', 'metric'])['value'].quantile(0.95).reset_index()
global_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

# Filter data within the alert time window
metrics_in_window = metrics_filtered[
    (metrics_filtered['time'] >= alert_start_us) & (metrics_filtered['time'] <= alert_end_us)
]

global_thresholds, metrics_in_window

```
Out[3]:
```
The global thresholds (P95) for all KPIs of the `checkout` service and its dependencies have been calculated. These thresholds provide a benchmark for identifying anomalies. Additionally, the metrics data has been filtered to include only the time window from 2026-04-25 13:18:12 to 2026-04-25 13:28:12 (UTC+8) for further analysis. This filtered data contains 3,452 rows of relevant metrics for the affected services and operations.

The original code execution output of IPython Kernel is also provided below for reference:

(                                      entity_name                       metric  p95_threshold
0                      accounting::orders receive  avg_request_latency_seconds   2.193581e-02
1                      accounting::orders receive                request_count   1.837140e+04
2                                            cart                          cpu   4.410636e-01
3                                            cart                      latency   4.956993e-03
4                                            cart                          mem   1.086076e+08
..                                            ...                          ...            ...
193   shipping::oteldemo.ShippingService/GetQuote  avg_request_latency_seconds   5.570722e-03
194   shipping::oteldemo.ShippingService/GetQuote                request_count   1.836500e+04
195   shipping::oteldemo.ShippingService/GetQuote                   slow_count   1.000000e+00
196  shipping::oteldemo.ShippingService/ShipOrder  avg_request_latency_seconds   1.028687e-05
197  shipping::oteldemo.ShippingService/ShipOrder                request_count   1.836500e+04

[198 rows x 3 columns],                    time domain     entity_set                         entity_id               entity_name       metric        value                            metric_set_id service
59022  1777094322000000    apm  apm.operation  c6388887e91aa969bdf210fe96a48098            frontend::POST  error_count  2374.000000  apm@metric_set@apm.metric.apm.operation    None
59023  1777094352000000    apm  apm.operation  c6388887e91aa969bdf210fe96a48098            frontend::POST  error_count  6885.000000  apm@metric_set@apm.metric.apm.operation    None
59024  1777094382000000    apm  apm.operation  c6388887e91aa969bdf210fe96a48098            frontend::POST  error_count  8898.000000  apm@metric_set@apm.metric.apm.operation    None
59025  1777094412000000    apm  apm.operation  c6388887e91aa969bdf210fe96a48098            frontend::POST  error_count  8822.000000  apm@metric_set@apm.metric.apm.operation    None
59026  1777094442000000    apm  apm.operation  c6388887e91aa969bdf210fe96a48098            frontend::POST  error_count  8757.000000  apm@metric_set@apm.metric.apm.operation    None
...                 ...    ...            ...                               ...                       ...          ...          ...                                      ...     ...
92147  1777094772000000    apm   apm.instance  4dab800a1af4e1c5967569ece76ece14  payment-696d688896-klnbs   error_rate     0.500662   apm@metric_set@apm.metric.apm.instance    None
92148  1777094802000000    apm   apm.instance  4dab800a1af4e1c5967569ece76ece14  payment-696d688896-klnbs   error_rate     0.506027   apm@metric_set@apm.metric.apm.instance    None
92149  1777094832000000    apm   apm.instance  4dab800a1af4e1c5967569ece76ece14  payment-696d688896-klnbs   error_rate     0.498722   apm@metric_set@apm.metric.apm.instance    None
92150  1777094862000000    apm   apm.instance  4dab800a1af4e1c5967569ece76ece14  payment-696d688896-klnbs   error_rate     0.492137   apm@metric_set@apm.metric.apm.instance    None
92151  1777094892000000    apm   apm.instance  4dab800a1af4e1c5967569ece76ece14  payment-696d688896-klnbs   error_rate     0.497726   apm@metric_set@apm.metric.apm.instance    None

[3452 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Merge the filtered metrics data with the global thresholds on entity_name and metric
metrics_with_thresholds = metrics_in_window.merge(
    global_thresholds, 
    on=['entity_name', 'metric'], 
    how='left'
)

# Identify anomalies where the metric value exceeds the P95 threshold
anomalies = metrics_with_thresholds[metrics_with_thresholds['value'] > metrics_with_thresholds['p95_threshold']]

# Display the anomalies
anomalies

```
Out[4]:
```
During the alert time window, 276 anomalies were detected where KPI values exceeded their respective P95 thresholds for the `checkout` service and its dependencies. Notable anomalies include high `error_count` for the `frontend::POST`, `frontend-proxy::ingress`, and `checkout::/oteldemo.CheckoutService/PlaceOrder` operations, as well as elevated `error_rate` for instances of the `payment` service. These anomalies indicate potential issues in these components during the incident.

The original code execution output of IPython Kernel is also provided below for reference:

time domain     entity_set                         entity_id                                     entity_name       metric        value                            metric_set_id service  p95_threshold
2     1777094382000000    apm  apm.operation  c6388887e91aa969bdf210fe96a48098                                  frontend::POST  error_count  8898.000000  apm@metric_set@apm.metric.apm.operation    None    8815.500000
3     1777094412000000    apm  apm.operation  c6388887e91aa969bdf210fe96a48098                                  frontend::POST  error_count  8822.000000  apm@metric_set@apm.metric.apm.operation    None    8815.500000
22    1777094382000000    apm  apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc                         frontend-proxy::ingress  error_count  8893.000000  apm@metric_set@apm.metric.apm.operation    None    8788.000000
23    1777094412000000    apm  apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc                         frontend-proxy::ingress  error_count  8820.000000  apm@metric_set@apm.metric.apm.operation    None    8788.000000
42    1777094382000000    apm  apm.operation  d219413245b68b297976412bbee076cf  checkout::/oteldemo.CheckoutService/PlaceOrder  error_count  8891.000000  apm@metric_set@apm.metric.apm.operation    None    8822.800000
...                ...    ...            ...                               ...                                             ...          ...          ...                                      ...     ...            ...
3411  1777094892000000    apm   apm.instance  d1f1b0349fadcfdc23ca3918ac448623                 frontend-proxy-67d64dc4f8-xz2fd   error_rate     0.122761   apm@metric_set@apm.metric.apm.instance    None       0.122511
3425  1777094712000000    apm   apm.instance  8161f0b2fe6bbf0d49bcadedf8f40935                        payment-696d688896-d2z22   error_rate     0.511371   apm@metric_set@apm.metric.apm.instance    None       0.509273
3431  1777094892000000    apm   apm.instance  8161f0b2fe6bbf0d49bcadedf8f40935                        payment-696d688896-d2z22   error_rate     0.509609   apm@metric_set@apm.metric.apm.instance    None       0.509273
3441  1777094592000000    apm   apm.instance  4dab800a1af4e1c5967569ece76ece14                        payment-696d688896-klnbs   error_rate     0.505198   apm@metric_set@apm.metric.apm.instance    None       0.505019
3448  1777094802000000    apm   apm.instance  4dab800a1af4e1c5967569ece76ece14                        payment-696d688896-klnbs   error_rate     0.506027   apm@metric_set@apm.metric.apm.instance    None       0.505019

[276 rows x 10 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Identify consecutive sub-series of anomalies for each component-KPI combination
anomalies['time'] = pd.to_datetime(anomalies['time'], unit='us').dt.tz_localize('UTC').dt.tz_convert('Asia/Shanghai')

# Sort anomalies by entity_name, metric, and time
anomalies_sorted = anomalies.sort_values(by=['entity_name', 'metric', 'time'])

# Group anomalies by component-KPI combination
grouped_anomalies = anomalies_sorted.groupby(['entity_name', 'metric'])

# Initialize a list to store identified faults
faults = []

# Iterate through each group to identify consecutive sub-series of anomalies
for (entity_name, metric), group in grouped_anomalies:
    group = group.reset_index(drop=True)
    group['time_diff'] = group['time'].diff().dt.total_seconds()
    
    # Define a threshold for consecutive anomalies (e.g., within 1 minute)
    consecutive_threshold = 60  # seconds
    
    # Identify consecutive anomalies
    group['is_consecutive'] = (group['time_diff'] <= consecutive_threshold) | (group['time_diff'].isna())
    group['fault_group'] = group['is_consecutive'].cumsum()
    
    # Filter out isolated noise spikes and minimal threshold breaches
    for fault_group, fault_data in group.groupby('fault_group'):
        if len(fault_data) > 1:  # Exclude isolated spikes
            max_breach = (fault_data['value'] - fault_data['p95_threshold']).max()
            if max_breach > 0.5 * fault_data['p95_threshold'].max():  # Exclude minimal breaches
                faults.append({
                    'entity_name': entity_name,
                    'metric': metric,
                    'start_time': fault_data['time'].min(),
                    'end_time': fault_data['time'].max(),
                    'occurrences': len(fault_data)
                })

# Convert faults to a DataFrame for display
faults_df = pd.DataFrame(faults)

faults_df

```
Out[5]:
```
No consecutive sub-series of anomalies (faults) were identified for any component-KPI combination in the filtered metrics data. This indicates that the detected anomalies were either isolated noise spikes or minimal threshold breaches that did not meet the criteria for significant faults.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: []
Index: []```
```

In [ ]:
# In[6]:

# Ensure startTime and endTime are integers
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Filter traces related to the checkout operation during the alert time window
checkout_traces = traces_data[
    (traces_data['spanName'] == 'checkout::/oteldemo.CheckoutService/PlaceOrder') &
    (traces_data['startTime'] >= alert_start_ns) &
    (traces_data['endTime'] <= alert_end_ns)
]

# Analyze the traces for errors or anomalies in the call chain
checkout_traces_with_errors = checkout_traces[
    (checkout_traces['statusCode'] != 0) | 
    (checkout_traces['statusMessage'].str.contains('error', case=False, na=False))
]

# Identify the most downstream faulty service
downstream_faulty_services = checkout_traces_with_errors.groupby('serviceName').size().reset_index(name='error_count')
downstream_faulty_services = downstream_faulty_services.sort_values(by='error_count', ascending=False)

checkout_traces_with_errors, downstream_faulty_services

```
Out[6]:
```
No traces related to the `checkout::/oteldemo.CheckoutService/PlaceOrder` operation during the alert time window contained errors or anomalies in the call chain. This indicates that there were no detectable issues in the traces for this operation during the specified period.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, events, __topic__, __tag__:__pack_id__, ip, links]
Index: []

[0 rows x 21 columns], Empty DataFrame
Columns: [serviceName, error_count]
Index: [])```
```

In [ ]:
# In[7]:

# Correct the timezone handling by using tz_convert instead of tz_localize
# Load the logs.parquet file
logs_file_path = 'rca100/cases/t001/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Filter logs for the checkout service and its dependencies during the alert time window
logs_filtered = logs_data[
    logs_data['_pod_name_'].str.contains('|'.join(relevant_services), case=False, na=False) &
    (pd.to_datetime(logs_data['_time_'], errors='coerce').dt.tz_convert('Asia/Shanghai') >= alert_start) &
    (pd.to_datetime(logs_data['_time_'], errors='coerce').dt.tz_convert('Asia/Shanghai') <= alert_end)
]

# Search for error messages or critical information in the content field
error_logs = logs_filtered[logs_filtered['content'].str.contains('error|critical|exception', case=False, na=False)]

error_logs

```
Out[7]:
```
The logs related to the `checkout` service and its dependencies during the alert time window contain 7,131 entries with error messages or critical information. Notable errors include repeated messages about failing to charge a card due to connection issues with the payment service. This indicates a potential issue with the payment service during the incident.

The original code execution output of IPython Kernel is also provided below for reference:

content                               _time_ _source_ _container_ip_                                       _image_name_  ...    __tag__:__pack_id__               __tag__:_cluster_id_    __tag__:__hostname__     __tag__:_node_name_ __tag__:_node_ip_
338900  {"level":"warn","time":1777094293959,"pid":18,...  2026-04-25T13:18:13.960189132+08:00   stdout    10.0.17.252  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  825F2AD29F3FCFFA-DF29  cfbbc0eabc19d43c0a6fb6889b4451ad0  cn-hongkong.10.0.1.107  cn-hongkong.10.0.1.107        10.0.1.107
338904  {"level":"warn","time":1777094293978,"pid":18,...  2026-04-25T13:18:13.979344281+08:00   stdout    10.0.17.252  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  825F2AD29F3FCFFA-DF29  cfbbc0eabc19d43c0a6fb6889b4451ad0  cn-hongkong.10.0.1.107  cn-hongkong.10.0.1.107        10.0.1.107
338906  {"level":"warn","time":1777094294028,"pid":18,...  2026-04-25T13:18:14.028711192+08:00   stdout    10.0.17.252  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  825F2AD29F3FCFFA-DF29  cfbbc0eabc19d43c0a6fb6889b4451ad0  cn-hongkong.10.0.1.107  cn-hongkong.10.0.1.107        10.0.1.107
338908  {"level":"warn","time":1777094294030,"pid":18,...  2026-04-25T13:18:14.030364788+08:00   stdout    10.0.17.252  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  825F2AD29F3FCFFA-DF29  cfbbc0eabc19d43c0a6fb6889b4451ad0  cn-hongkong.10.0.1.107  cn-hongkong.10.0.1.107        10.0.1.107
338912  {"level":"warn","time":1777094294043,"pid":18,...  2026-04-25T13:18:14.044083114+08:00   stdout    10.0.17.252  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  825F2AD29F3FCFFA-DF29  cfbbc0eabc19d43c0a6fb6889b4451ad0  cn-hongkong.10.0.1.107  cn-hongkong.10.0.1.107        10.0.1.107
...                                                   ...                                  ...      ...            ...                                                ...  ...                    ...                                ...                     ...                     ...               ...
533989    details: 'failed to charge card: could not c...  2026-04-25T13:28:12.675542952+08:00   stderr      10.0.2.33  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  E097875A1D2189CF-B774  cfbbc0eabc19d43c0a6fb6889b4451ad0   cn-hongkong.10.0.2.35   cn-hongkong.10.0.2.35         10.0.2.35
533992  Error: 13 INTERNAL: failed to charge card: cou...  2026-04-25T13:28:12.683898275+08:00   stderr      10.0.2.33  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  E097875A1D2189CF-B774  cfbbc0eabc19d43c0a6fb6889b4451ad0   cn-hongkong.10.0.2.35   cn-hongkong.10.0.2.35         10.0.2.35
533996    details: 'failed to charge card: could not c...  2026-04-25T13:28:12.683919848+08:00   stderr      10.0.2.33  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  E097875A1D2189CF-B774  cfbbc0eabc19d43c0a6fb6889b4451ad0   cn-hongkong.10.0.2.35   cn-hongkong.10.0.2.35         10.0.2.35
533999   ⨯ Error: 13 INTERNAL: failed to charge card: ...  2026-04-25T13:28:12.684418333+08:00   stderr      10.0.2.33  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  E097875A1D2189CF-B774  cfbbc0eabc19d43c0a6fb6889b4451ad0   cn-hongkong.10.0.2.35   cn-hongkong.10.0.2.35         10.0.2.35
534003    details: 'failed to charge card: could not c...  2026-04-25T13:28:12.684437855+08:00   stderr      10.0.2.33  o11y-demo-registry-vpc.cn-hongkong.cr.aliyuncs...  ...  E097875A1D2189CF-B774  cfbbc0eabc19d43c0a6fb6889b4451ad0   cn-hongkong.10.0.2.35   cn-hongkong.10.0.2.35         10.0.2.35

[7131 rows x 15 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```